In [2]:
import pandas as pd
import numpy as np

# ============================================================
# STEP 1: Load Dataset
# ============================================================

file_path = "Cleaned_Activation_Onboarding_Funnel.xlsx"

df = pd.read_excel(file_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)


# ============================================================
# STEP 2: Understand Dataset
# ============================================================

print("\nFirst 5 Rows:")
print(df.head())

print("\nDataset Information:")
print(df.info())

print("\nDescriptive Statistics:")
print(df.describe())


# ============================================================
# STEP 3: Data Quality Check
# ============================================================

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())


# ============================================================
# STEP 4: Basic Business Metrics
# ============================================================

total_users = df["user_id"].nunique()

activated_users = df.loc[
    df["activated"] == 1, "user_id"
].nunique()

total_revenue = df["revenue"].sum()

activation_rate = (
    activated_users / total_users
) * 100

print("\nBusiness Metrics")
print("----------------")
print("Total Users:", total_users)
print("Activated Users:", activated_users)
print("Activation Rate:", round(activation_rate, 2), "%")
print("Total Revenue:", round(total_revenue, 2))


# ============================================================
# STEP 5: Build Funnel Summary
# ============================================================

funnel = pd.DataFrame({
    "stage": [
        "Signup",
        "Profile",
        "Search",
        "Listing",
        "Booking",
        "Payment",
        "Activation"
    ],
    "users": [
        df["signup_completed"].sum(),
        df["profile_completed"].sum(),
        df["search_completed"].sum(),
        df["listing_completed"].sum(),
        df["booking_completed"].sum(),
        df["payment_completed"].sum(),
        df["activated"].sum()
    ]
})

funnel["conversion_rate"] = (
    funnel["users"] / total_users * 100
)

funnel["drop_off_users"] = (
    funnel["users"].shift(1) - funnel["users"]
)

funnel.loc[0, "drop_off_users"] = 0

funnel["drop_off_rate"] = (
    funnel["drop_off_users"]
    / funnel["users"].shift(1)
    * 100
)

funnel.loc[0, "drop_off_rate"] = 0

print("\nActivation Funnel:")
print(funnel)


# ============================================================
# STEP 6: Revenue by Funnel Stage
# ============================================================

stage_revenue = pd.DataFrame({
    "stage": [
        "Signup",
        "Profile",
        "Search",
        "Listing",
        "Booking",
        "Payment",
        "Activation"
    ],
    "users": funnel["users"],
})

stage_revenue["potential_revenue"] = (
    stage_revenue["users"]
    * df["revenue"].mean()
)

print("\nRevenue Opportunity by Stage:")
print(stage_revenue)


# ============================================================
# STEP 7: Marketplace Side Analysis
# ============================================================

side_analysis = df.groupby(
    "marketplace_side"
).agg(
    users=("user_id", "nunique"),
    profiles=("profile_completed", "sum"),
    searches=("search_completed", "sum"),
    listings=("listing_completed", "sum"),
    bookings=("booking_completed", "sum"),
    payments=("payment_completed", "sum"),
    activated=("activated", "sum"),
    revenue=("revenue", "sum")
).reset_index()

side_analysis["activation_rate"] = (
    side_analysis["activated"]
    / side_analysis["users"]
    * 100
)

print("\nMarketplace Side Analysis:")
print(side_analysis)


# ============================================================
# STEP 8: Device Analysis
# ============================================================

device_analysis = df.groupby(
    "device"
).agg(
    users=("user_id", "nunique"),
    activated_users=("activated", "sum"),
    revenue=("revenue", "sum")
).reset_index()

device_analysis["activation_rate"] = (
    device_analysis["activated_users"]
    / device_analysis["users"]
    * 100
)

print("\nDevice Analysis:")
print(device_analysis)


# ============================================================
# STEP 9: Acquisition Source Analysis
# ============================================================

source_analysis = df.groupby(
    "acquisition_source"
).agg(
    users=("user_id", "nunique"),
    activated_users=("activated", "sum"),
    revenue=("revenue", "sum")
).reset_index()

source_analysis["activation_rate"] = (
    source_analysis["activated_users"]
    / source_analysis["users"]
    * 100
)

print("\nAcquisition Source Analysis:")
print(source_analysis)


# ============================================================
# STEP 10: Funnel Analysis by Marketplace Side
# ============================================================

side_funnel = df.groupby(
    "marketplace_side"
).agg(
    signup=("signup_completed", "sum"),
    profile=("profile_completed", "sum"),
    search=("search_completed", "sum"),
    listing=("listing_completed", "sum"),
    booking=("booking_completed", "sum"),
    payment=("payment_completed", "sum"),
    activation=("activated", "sum")
).reset_index()

print("\nFunnel by Marketplace Side:")
print(side_funnel)


# ============================================================
# STEP 11: Drop-Off Analysis
# ============================================================

funnel["previous_stage_users"] = funnel["users"].shift(1)

funnel["drop_off_users"] = (
    funnel["previous_stage_users"]
    - funnel["users"]
)

funnel.loc[0, "drop_off_users"] = 0

funnel["drop_off_percentage"] = np.where(
    funnel["previous_stage_users"] > 0,
    funnel["drop_off_users"]
    / funnel["previous_stage_users"]
    * 100,
    0
)

print("\nDrop-Off Analysis:")
print(
    funnel[
        [
            "stage",
            "users",
            "drop_off_users",
            "drop_off_percentage"
        ]
    ]
)


# ============================================================
# STEP 12: Revenue Opportunity
# ============================================================

average_revenue = df.loc[
    df["revenue"] > 0,
    "revenue"
].mean()

funnel["revenue_opportunity"] = (
    funnel["drop_off_users"]
    * average_revenue
)

print("\nRevenue Opportunity:")
print(
    funnel[
        [
            "stage",
            "drop_off_users",
            "revenue_opportunity"
        ]
    ]
)


# ============================================================
# STEP 13: Highest Drop-Off Stage
# ============================================================

highest_dropoff = funnel.loc[
    funnel["drop_off_users"].idxmax()
]

print("\nHighest Drop-Off Stage")
print("----------------------")
print("Stage:", highest_dropoff["stage"])
print("Users Lost:", int(highest_dropoff["drop_off_users"]))
print(
    "Drop-Off Rate:",
    round(highest_dropoff["drop_off_percentage"], 2),
    "%"
)


# ============================================================
# STEP 14: Highest Revenue Opportunity
# ============================================================

highest_revenue_opportunity = funnel.loc[
    funnel["revenue_opportunity"].idxmax()
]

print("\nHighest Revenue Opportunity")
print("---------------------------")
print(
    "Stage:",
    highest_revenue_opportunity["stage"]
)
print(
    "Revenue Opportunity:",
    round(
        highest_revenue_opportunity[
            "revenue_opportunity"
        ],
        2
    )
)


# ============================================================
# STEP 15: Activation by Acquisition Source
# ============================================================

source_analysis = source_analysis.sort_values(
    "activation_rate",
    ascending=False
)

print("\nActivation by Acquisition Source:")
print(source_analysis)


# ============================================================
# STEP 16: Activation by Device
# ============================================================

device_analysis = device_analysis.sort_values(
    "activation_rate",
    ascending=False
)

print("\nActivation by Device:")
print(device_analysis)


# ============================================================
# STEP 17: Activation by Marketplace Side
# ============================================================

side_analysis = side_analysis.sort_values(
    "activation_rate",
    ascending=False
)

print("\nActivation by Marketplace Side:")
print(side_analysis)


# ============================================================
# STEP 18: Business Insights
# ============================================================

print("\nKEY BUSINESS INSIGHTS")
print("=====================")

print(
    f"1. Total users analyzed: {total_users:,}"
)

print(
    f"2. Activated users: {activated_users:,}"
)

print(
    f"3. Overall activation rate: "
    f"{activation_rate:.2f}%"
)

print(
    f"4. Total recorded revenue: "
    f"{total_revenue:,.2f}"
)

print(
    f"5. Largest user drop-off occurs at: "
    f"{highest_dropoff['stage']}"
)

print(
    f"6. Users lost at that stage: "
    f"{int(highest_dropoff['drop_off_users']):,}"
)

print(
    f"7. Largest calculated revenue opportunity "
    f"is associated with: "
    f"{highest_revenue_opportunity['stage']}"
)

print(
    "8. Activation was compared across marketplace sides."
)

print(
    "9. Activation was compared across devices."
)

print(
    "10. Acquisition sources were analyzed for activation "
    "and revenue differences."
)


# ============================================================
# STEP 19: Save Analysis Results
# ============================================================

with pd.ExcelWriter(
    "Python Analysis Result.xlsx",
    engine="openpyxl"
) as writer:

    funnel.to_excel(
        writer,
        sheet_name="Funnel Analysis",
        index=False
    )

    stage_revenue.to_excel(
        writer,
        sheet_name="Revenue Opportunity",
        index=False
    )

    side_analysis.to_excel(
        writer,
        sheet_name="Side Analysis",
        index=False
    )

    side_funnel.to_excel(
        writer,
        sheet_name="Side Funnel",
        index=False
    )

    device_analysis.to_excel(
        writer,
        sheet_name="Device Analysis",
        index=False
    )

    source_analysis.to_excel(
        writer,
        sheet_name="Source Analysis",
        index=False
    )

print("\nPython Analysis Result.xlsx created successfully.")

Dataset loaded successfully.
Shape: (1800, 21)

First 5 Rows:
  user_id signup_date marketplace_side   device acquisition_source  \
0  U10001  2025-04-13           Supply  Android        Paid Search   
1  U10002  2025-06-29           Supply  Android           Referral   
2  U10003  2025-04-03           Supply      Web            Organic   
3  U10004  2025-01-15           Demand  Android        Paid Search   
4  U10005  2025-04-17           Demand      Web            Organic   

   signup_completed  profile_completed  search_completed  listing_completed  \
0                 1                  1                 0                  1   
1                 1                  1                 1                  1   
2                 1                  1                 0                  1   
3                 1                  1                 1                  0   
4                 1                  0                 0                  0   

   booking_completed  ...  activated  reve